In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 30. ZeRO and FSDP — Extreme Memory Optimization

> **Learning Goals**
> - Understand what ZeRO Stage 1/2/3 shard
> - Understand how FSDP (Fully Sharded Data Parallel) works
> - Quantitatively compute memory savings

## 30.1 Memory Limits of DDP

DDP keeps a full model copy on every GPU. A 70B model cannot fit on a single GPU.

ZeRO's philosophy: **"shard everything"**: optimizer states, gradients, and parameters.

## 30.2 Memory Breakdown

Training memory = parameters + gradients + optimizer states + activations

For AdamW:
- Parameters $P$ (FP16): $2P$ bytes
- Gradients (FP16): $2P$ bytes
- Optimizer states (FP32: m, v, master): $12P$ bytes
- Total excluding activations: $16P$ bytes

7B model: 112GB, too large for one GPU.

## 30.3 ZeRO Stage 1 — Shard Optimizer States

Shard optimizer states across $N$ GPUs:
- Memory: $16P/N + 4P$ (parameters and gradients remain replicated)
- 7B, N=8: $16P/8 + 4P = 6P = 42GB$, still large

## 30.4 ZeRO Stage 2 — Shard Gradients Too

Shard gradients as well:
- Memory: $16P/N + 2P = 4P$
- Communication uses Reduce-Scatter instead of All-Reduce

## 30.5 ZeRO Stage 3 — Shard Parameters Too

Shard everything:
- Memory: $16P/N$ + activations
- 7B, N=8: $2P + activations \approx 14GB$, feasible on one 40GB GPU
- Communication: All-Gather for forward, Reduce-Scatter for backward


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zero_memory_gb(model_params_b, n_gpus, stage, optimizer_factor=12, dtype_bytes=2):
    """ZeRO memory by ZeRO stage (GB).
    model_params_b: Parameter Count (unit: billion)
    optimizer_factor: for Adam 12 (m, v, master FP32)
    dtype_bytes: 2 for FP16
    """
    P = model_params_b  # billions
    # FP16 parameters: 2 bytes, FP16 grad: 2 bytes
    # Optimizer (FP32 m, v, master): 12 bytes per param
    param_mem = P * dtype_bytes  # GB
    grad_mem = P * dtype_bytes
    opt_mem = P * optimizer_factor

    if stage == 0:  # DDP
        return param_mem + grad_mem + opt_mem
    elif stage == 1:  # shard optimizer only
        return param_mem + grad_mem + opt_mem / n_gpus
    elif stage == 2:  # Optimizer + Gradient split
        return param_mem + (grad_mem + opt_mem) / n_gpus
    elif stage == 3:  # shard everything
        return (param_mem + grad_mem + opt_mem) / n_gpus

# Visualization: 7B Model, by num of GPUs
model_size = 7  # 7B
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: by num of GPUs
ax = axes[0]
for stage, label, color in [(0, 'DDP', 'blue'), (1, 'ZeRO-1', 'green'),
                             (2, 'ZeRO-2', 'orange'), (3, 'ZeRO-3', 'red')]:
    mems = [zero_memory_gb(model_size, n, stage) for n in n_gpus_list]
    ax.plot(n_gpus_list, mems, 'o-', label=label, linewidth=2, color=color)
ax.axhline(40, color='gray', linestyle='--', label='A100 40GB')
ax.axhline(80, color='gray', linestyle=':', label='A100 80GB')
ax.set_xlabel('GPU num')
ax.set_ylabel('per GPU Memory (GB)')
ax.set_title(f'Memory by ZeRO Stage (7B Model)')
ax.legend(); ax.grid(True, alpha=0.3)

# right: by model size
ax = axes[1]
for stage, label, color in [(0, 'DDP', 'blue'), (3, 'ZeRO-3', 'red')]:
    for n in [1, 8, 32]:
        sizes = [1, 7, 13, 30, 70, 175]
        mems = [zero_memory_gb(s, n, stage) for s in sizes]
        ax.plot(sizes, mems, 'o-', label=f'{label}, N={n}', linewidth=2)
ax.set_xlabel('Model Size (B params)')
ax.set_ylabel('per GPU Memory (GB)')
ax.set_title('Memory by Model Size')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch30_zero_memory.png', dpi=100, bbox_inches='tight')
plt.show()

# numeric output
print(f"\n{'model':>6} | {'GPU':>4} | {'DDP':>8} | {'ZeRO-1':>8} | {'ZeRO-2':>8} | {'ZeRO-3':>8}")
print("-" * 55)
for size in [7, 13, 70]:
    for n in [1, 8]:
        ddp = zero_memory_gb(size, n, 0)
        z1 = zero_memory_gb(size, n, 1)
        z2 = zero_memory_gb(size, n, 2)
        z3 = zero_memory_gb(size, n, 3)
        print(f"{size:>5}B | {n:>4} | {ddp:>7.1f}G | {z1:>7.1f}G | {z2:>7.1f}G | {z3:>7.1f}G")


## 30.6 PyTorch FSDP

FSDP (Fully Sharded Data Parallel) is PyTorch's ZeRO-3 implementation.

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyLargeModel()
model = FSDP(model)  # automatically shards parameters
```

Workflow:
1. Before forward: All-Gather parameters
2. After forward: release parameter memory
3. Backward: All-Gather again and compute gradients
4. Reduce-Scatter gradients

## 30.7 CPU Offloading

When GPU memory is insufficient, move optimizer states to CPU RAM.
- ZeRO-Infinity and FSDP with CPU offload
- Slower, but enables training larger models


In [ ]:
# FSDP simulation (gradient sharding)
import torch
import torch.nn as nn
import torch.nn.functional as F

class FSDPSimulator:
    """Simulator that mimics FSDP on a single GPU."""
    def __init__(self, model, n_shards=4):
        self.model = model
        self.n_shards = n_shards
        # Split parameters into n_shards; each synthetic GPU owns one shard
        params = list(model.parameters())
        self.shards = [params[i::n_shards] for i in range(n_shards)]

    def step(self, x, y, optimizer):
        """one step: Forward Pass → Backward Pass → update each shard."""
        optimizer.zero_grad()
        out = self.model(x)
        loss = F.mse_loss(out, y)
        loss.backward()

        # use only each shard gradient (other shards set to zero)
        # Real FSDP synchronizes by communication; the simulation uses all gradients
        optimizer.step()
        return loss.item()

# simulation
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(128, 512), nn.ReLU(),
    nn.Linear(512, 128)
)
fsdp = FSDPSimulator(model, n_shards=4)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(8, 128)
y = torch.randn(8, 128)
loss = fsdp.step(x, y, opt)
print(f"FSDP simulation loss: {loss:.4f}")
print(f"total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"num of shards: {fsdp.n_shards} (per shard ~{sum(p.numel() for p in model.parameters()) // 4:,} parameters)")


## 30.8 Key Takeaways

| Stage | Sharded State | Memory | Communication |
|---|---|---|---|
| DDP | none | $16P$ | All-Reduce |
| ZeRO-1 | optimizer | $16P/N + 4P$ | All-Reduce |
| ZeRO-2 | + gradients | $16P/N + 2P$ | Reduce-Scatter |
| ZeRO-3 | + parameters | $16P/N$ | All-Gather + Reduce-Scatter |
| FSDP | ZeRO-3 (PyTorch) | $16P/N$ | same |
| ZeRO-Infinity | + CPU/NVMe | lower | slower |

## Exercises

1. Compute memory for ZeRO Stage 0/1/2/3 on a 70B model with N=16.
2. Explain why FSDP performs All-Gather pre the forward pass.
3. Compare ZeRO-3 communication overhead with DDP.
4. Explain why CPU offloading slows training.
5. Compute how many A100 80GB GPUs are needed to train a 70B model with ZeRO-3.

> Solutions: `solutions/ch30_solutions.ipynb`
